# Train a fresh Theta NNUE

This notebook trains Theta's custom NNUE format from labeled positions. It can use Stockfish NNUE scores as a teacher without copying the Stockfish net into Theta.

Use a labeled CSV with `fen,score_cp`, or upload a Stockfish executable so the notebook can create labels. Scores must be from the side-to-move perspective.

In [ ]:
%pip -q install python-chess tqdm pandas numpy

In [ ]:
from pathlib import Path
import os
import random
import shutil
import struct
import subprocess

import chess
import chess.engine
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

SEED = 20260802
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_PATH = None
STOCKFISH_PATH = None
STOCKFISH_EVAL_FILE = None
NUM_POSITIONS = 50000
MAX_PLIES = 100
LABEL_DEPTH = 8
EPOCHS = 8
BATCH_SIZE = 2048
OUTPUT_PATH = Path('/kaggle/working/theta.nnue')
LABELS_PATH = Path('/kaggle/working/theta_labels.csv')
INPUT_FEATURES = 780
HIDDEN_SIZE = 64
OUTPUT_HIDDEN_SIZE = 32
OUTPUT_SCALE = 1000.0
SCORE_CLIP = 1500.0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Find input data

For stronger training, upload a large CSV or PGN dataset as a Kaggle input. If `DATA_PATH` is left as `None`, the notebook samples legal positions and labels them with `STOCKFISH_PATH`. Set `STOCKFISH_EVAL_FILE` to use a specific Stockfish NNUE file.

In [ ]:
def find_file(names):
    roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path('.')]
    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob('*'):
            if path.is_file() and path.name.lower() in names:
                return path
    return None

if DATA_PATH is None:
    DATA_PATH = find_file({'labels.csv', 'theta_labels.csv'})
if STOCKFISH_PATH is None:
    STOCKFISH_PATH = shutil.which('stockfish')
    if STOCKFISH_PATH is None:
        path = find_file({'stockfish', 'stockfish.exe', 'stockfish-18.exe'})
        STOCKFISH_PATH = str(path) if path else None

print('data:', DATA_PATH)
print('stockfish:', STOCKFISH_PATH)
print('stockfish eval file:', STOCKFISH_EVAL_FILE)

In [ ]:
def sample_fens(count, max_plies, seed):
    rng = random.Random(seed)
    fens = []
    while len(fens) < count:
        board = chess.Board()
        plies = rng.randrange(0, max_plies + 1)
        for _ in range(plies):
            if board.is_game_over():
                break
            moves = list(board.legal_moves)
            if not moves:
                break
            board.push(rng.choice(moves))
        if not board.is_game_over():
            fens.append(board.fen())
    return fens

def load_or_label():
    if DATA_PATH is not None:
        frame = pd.read_csv(DATA_PATH)
        if 'score_cp' not in frame.columns and 'score' in frame.columns:
            frame = frame.rename(columns={'score': 'score_cp'})
        if 'fen' not in frame.columns or 'score_cp' not in frame.columns:
            raise ValueError('CSV needs fen and score_cp columns')
        return frame[['fen', 'score_cp']].dropna().reset_index(drop=True)
    if STOCKFISH_PATH is None:
        raise RuntimeError('Set STOCKFISH_PATH or upload a labels.csv file')
    fens = sample_fens(NUM_POSITIONS, MAX_PLIES, SEED)
    rows = []
    engine = chess.engine.SimpleEngine.popen_uci(str(STOCKFISH_PATH))
    try:
        engine.configure({'Threads': 1, 'Hash': 64})
        if STOCKFISH_EVAL_FILE is not None:
            engine.configure({'EvalFile': str(STOCKFISH_EVAL_FILE)})
        for fen in tqdm(fens, desc='labeling'):
            board = chess.Board(fen)
            info = engine.analyse(board, chess.engine.Limit(depth=LABEL_DEPTH))
            score = info['score'].pov(board.turn).score(mate_score=10000)
            rows.append((fen, int(score)))
    finally:
        engine.quit()
    frame = pd.DataFrame(rows, columns=['fen', 'score_cp'])
    frame.to_csv(LABELS_PATH, index=False)
    return frame

labels = load_or_label()
labels['score_cp'] = labels['score_cp'].clip(-SCORE_CLIP, SCORE_CLIP)
print('positions:', len(labels))
labels.head()

## Encode Theta features

Theta uses rank 8 as square row zero. Piece colors are relative to the side to move, so the network output is always from the side-to-move perspective.

In [ ]:
def theta_square(square):
    return (7 - chess.square_rank(square)) * 8 + chess.square_file(square)

def feature_ids(board):
    ids = []
    for square, piece in board.piece_map().items():
        relative_color = 0 if piece.color == board.turn else 1
        piece_type = piece.piece_type - 1
        ids.append((relative_color * 6 + piece_type) * 64 + theta_square(square))
    rights = [
        (board.turn, board.has_kingside_castling_rights(board.turn)),
        (board.turn, board.has_queenside_castling_rights(board.turn)),
        (not board.turn, board.has_kingside_castling_rights(not board.turn)),
        (not board.turn, board.has_queenside_castling_rights(not board.turn)),
    ]
    ids.extend(768 + index for index, (_, present) in enumerate(rights) if present)
    if board.ep_square is not None:
        ids.append(772 + chess.square_file(board.ep_square))
    return ids

def encode_fens(fens):
    encoded = np.zeros((len(fens), INPUT_FEATURES), dtype=np.float32)
    for row, fen in enumerate(fens):
        encoded[row, feature_ids(chess.Board(fen))] = 1.0
    return encoded

features = encode_fens(labels['fen'].tolist())
targets = labels['score_cp'].to_numpy(np.float32) / OUTPUT_SCALE
print('features:', features.shape, 'targets:', targets.shape)

In [ ]:
class ThetaNNUE(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(INPUT_FEATURES, HIDDEN_SIZE)
        self.fc2 = nn.Linear(HIDDEN_SIZE, OUTPUT_HIDDEN_SIZE)
        self.out = nn.Linear(OUTPUT_HIDDEN_SIZE, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.out(x).squeeze(-1)

model = ThetaNNUE().to(device)
order = np.random.default_rng(SEED).permutation(len(features))
split = max(1, int(len(order) * 0.9))
train_order, valid_order = order[:split], order[split:]
train_data = TensorDataset(
    torch.from_numpy(features[train_order]),
    torch.from_numpy(targets[train_order]),
)
valid_data = TensorDataset(
    torch.from_numpy(features[valid_order]),
    torch.from_numpy(targets[valid_order]),
) if len(valid_order) else None
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=device.type == 'cuda')
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False) if valid_data else None
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.SmoothL1Loss(beta=0.05)

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(batch_x)
    train_loss /= len(train_data)
    valid_loss = float('nan')
    if valid_loader is not None:
        model.eval()
        total = 0.0
        count = 0
        with torch.no_grad():
            for batch_x, batch_y in valid_loader:
                prediction = model(batch_x.to(device))
                total += loss_fn(prediction, batch_y.to(device)).item() * len(batch_x)
                count += len(batch_x)
        valid_loss = total / count
    print(f'epoch {epoch + 1}: train {train_loss:.6f}, valid {valid_loss:.6f}')

## Export Theta's binary format

The first-layer weights are transposed during export so C++ can add one 64-float column per active feature.

In [ ]:
MAGIC = b'THNNUE01'
VERSION = 1

def export_theta(model, path):
    model = model.to('cpu').eval()
    with torch.no_grad():
        w1 = model.fc1.weight.numpy().T.astype('<f4')
        b1 = model.fc1.bias.numpy().astype('<f4')
        w2 = model.fc2.weight.numpy().T.astype('<f4')
        b2 = model.fc2.bias.numpy().astype('<f4')
        wo = model.out.weight.numpy().reshape(-1).astype('<f4')
        bo = np.float32(model.out.bias.item())
    with open(path, 'wb') as handle:
        handle.write(struct.pack('<8s5If', MAGIC, VERSION, INPUT_FEATURES, HIDDEN_SIZE, OUTPUT_HIDDEN_SIZE, 0, OUTPUT_SCALE))
        w1.tofile(handle)
        b1.tofile(handle)
        w2.tofile(handle)
        b2.tofile(handle)
        wo.tofile(handle)
        np.asarray([bo], dtype='<f4').tofile(handle)
    return path

export_theta(model, OUTPUT_PATH)
print('wrote:', OUTPUT_PATH, 'bytes:', OUTPUT_PATH.stat().st_size)

In [ ]:
@torch.no_grad()
def sparse_prediction(model, row):
    active = torch.from_numpy(np.flatnonzero(row)).long()
    first = model.fc1.bias + model.fc1.weight[:, active].sum(dim=1)
    second = torch.relu(model.fc2(first))
    return model.out(second).item()

sample_count = min(32, len(features))
dense = model(torch.from_numpy(features[:sample_count])).numpy()
sparse = np.asarray([sparse_prediction(model, row) for row in features[:sample_count]])
print('max parity error:', np.max(np.abs(dense - sparse)))
assert np.allclose(dense, sparse, atol=1e-5)
print('Theta export is ready:', OUTPUT_PATH)

## Use the model in Theta

Copy `theta.nnue` from the Kaggle output into a location visible to the engine, then send:

```text
setoption name NNUEFile value theta.nnue
setoption name Use NNUE value true
isready
```

Use deeper Stockfish NNUE labels and more varied positions for a stronger model. The included defaults are a smoke-test training run, not a finished 2700-strength net.